# Session 8 · Exercises

*Course material, licence CC BY 4.0.*

Work in pairs. The lecturer chooses which exercises to set; there are more than fit into 30 minutes. *Paper* and *Predict* tasks are answered in the markdown cell below the task, before any code is run; *Code* tasks in the empty code cells. Exercises marked *Stretch* are for pairs who finish early.

## Setup

Run these cells once before the exercises that use data.

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

# The IBM Telco churn data, as in the lecture notebooks of this session (downloaded from GitHub)
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)
y = (telco["Churn"] == "Yes").astype(int)                  # 1 = customer left
X = telco.drop(columns=["customerID", "Churn"])
numeric = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical = [c for c in X.columns if c not in numeric]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print(len(X_train), "training and", len(X_test), "test customers;", round(y.mean(), 3), "churn rate")

## Part 1 · Classifiers, pipelines and the confusion matrix

### 1.1 · k-NN on graph paper

*Paper, 10 min*

> Six training points in two features: class A at (1, 1), (2, 1) and (1, 2); class B at (4, 4), (5, 4) and (3, 3). Classify the new point (2.5, 2.5) with k = 1 and with k = 3 (Euclidean distance).
>
> What is the training accuracy of 1-NN, and why is that suspicious?

*Source:* [k-nearest neighbors algorithm, Wikipedia](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm); the six points are an original example for this course

Your answer:

### 1.2 · Fill the confusion matrix from a story

*Paper, 10 min*

> 100 customers, 20 churn; the model flags 25, of whom 15 churn. Fill the confusion matrix and compute precision, recall and F1.

*Source:* original exercise for this course; background: [Precision and recall, Wikipedia](https://en.wikipedia.org/wiki/Precision_and_recall)

Your answer:

### 1.3 · The 99 % accurate medical test

*Paper, 10 min*

> Prevalence 1 %, sensitivity and specificity 99 %. You test positive: how likely are you ill?

*Source:* [Base rate fallacy, Wikipedia](https://en.wikipedia.org/wiki/Base_rate_fallacy)

Your answer:

### 1.4 · Does k-NN need the scaler?

*Code, 20 min*

> Use the setup data. Build two pipelines with a `ColumnTransformer` (one-hot encoding of the categorical columns) and `KNeighborsClassifier(n_neighbors=50)`: one that scales the numeric columns with `StandardScaler` and one that passes them through unscaled. Encode `Contract` with an `OrdinalEncoder` in the order month-to-month, one year, two year in both.
>
> Compare the two with 5-fold cross-validation (`cv`) on the training data: accuracy, recall of churners and macro-F1. Which numeric column dominates the distances without scaling, and why?

*Source:* original exercise for this course; background: [scikit-learn: Importance of feature scaling](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html)

## Part 2 · Scores, thresholds and rare classes

### 2.1 · 99 % accuracy

*Paper, 5 min*

> 1 % of transactions are fraud. A model that always says "no fraud" scores 99 %. Is it a good model? What accuracy does always predicting "no churn" reach on the Telco customers (1,869 of 7,043 churn)?

*Source:* [Accuracy paradox, Wikipedia](https://en.wikipedia.org/wiki/Accuracy_paradox); [IBM Telco churn data](https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv)

Your answer:

### 2.2 · AUC by counting pairs

*Paper, 10 min*

> Three churners with scores 0.8, 0.6 and 0.3, and three stayers with scores 0.7, 0.4 and 0.2. What share of churner–stayer pairs is ranked correctly? That share is the AUC.

*Source:* [Hanley & McNeil (1982), "The meaning and use of the area under a ROC curve", *Radiology*](https://doi.org/10.1148/radiology.143.1.7063747) (ties count ½); the six scores are an original example for this course

Your answer:

### 2.3 · Threshold from money

*Paper, 10 min*

> A retention offer costs €20 and keeps a churner half the time; a lost customer costs €200. From which churn probability is the offer worth it?

*Source:* [Elkan (2001), "The foundations of cost-sensitive learning", IJCAI](https://cseweb.ucsd.edu/~elkan/rescale.pdf)

Your answer:

### 2.4 · How calibrated are you?

*Class, 15 min*

> Ten questions about Berlin, such as "Is Berlin larger in area than Paris?". Each student answers and states a confidence from 50 % to 100 %. Plot the class's calibration curve.

*Source:* [Overconfidence effect, Wikipedia](https://en.wikipedia.org/wiki/Overconfidence_effect)

Your answer:

### 2.5 · Half-balanced, and a more valuable customer

*Code, 25 min*

> Use the setup data. Put a preprocessing step (one-hot encoding and scaling, as in the lecture), a sampler and a `LogisticRegression(max_iter=2000)` in an imbalanced-learn pipeline (`from imblearn.pipeline import make_pipeline`).
>
> (1) Compare `RandomOverSampler(sampling_strategy=0.5)` (churners raised to half the loyal customers) with full balancing (`sampling_strategy=1.0`) and with no sampler: precision, recall and F1 of the churners with out-of-fold predictions (`cross_val_predict` on the training data with `cv`).
> (2) A lost customer is now worth €1,000 instead of €400; an offer still costs €100. Compute the campaign cost of each model at threshold 0.5 on the out-of-fold predictions, and the cost of the uncorrected model at the threshold c / L. Which wins now?

*Source:* original exercise for this course; background: [imbalanced-learn: Common pitfalls and recommended practices](https://imbalanced-learn.org/stable/common_pitfalls.html)

### 2.6 · Say what the correction does

*Pairs, 5 min, *Stretch**

> For each strategy of 2.5, write one sentence in the form "recall of churners rose from … to …, precision fell from … to …, the campaign cost changed by …". Which sentence would you show the retention team?

*Source:* original exercise for this course

Your answer: